<a href="https://colab.research.google.com/github/thomaslu678/gee-test/blob/main/clean/11_ArcGis_REST_Forest_Inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import scipy.stats as stats
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point
import requests

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import shap

import seaborn as sns

In [ ]:
url = "https://www.arcgis.com/apps/mapviewer/index.html?webmap=5005408e8d2742eb830b9cee87833d4a"
separator = "webmap="
_, _, id = url.partition(separator)

In [ ]:
def get_operational_layer_urls(portal_base, webmap_id):
    url = f"{portal_base}/sharing/rest/content/items/{webmap_id}/data"
    params = {"f": "pjson"}

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    layers = data.get("operationalLayers", [])

    urls = []
    for layer in layers:
        if "url" in layer:
            urls.append(layer["url"])

    return urls

In [ ]:
def fetch_all_features(layer_url, page_size=2000):
    query_url = f"{layer_url}/query"

    all_features = []
    offset = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size
        }

        # IMPORTANT: use POST (not GET)
        r = requests.post(query_url, data=params)
        r.raise_for_status()
        data = r.json()

        features = data.get("features", [])

        if not features:
            break

        all_features.extend(features)
        offset += page_size

        print(f"Fetched {len(all_features)} features...")

        # Stop if we got fewer than requested (last page)
        if len(features) < page_size:
            break

    return all_features

In [ ]:
base_url = "https://slcgov.maps.arcgis.com/"

urls = get_operational_layer_urls(base_url, id)

In [ ]:
urls

['https://services.arcgis.com/mMBpeYj0vPFotzbe/arcgis/rest/services/Urban_Forestry_Inventory/FeatureServer/0']

In [ ]:
count_url = f"{urls[0]}/query"

count_params = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

r = requests.post(count_url, data=count_params)
print("Total features on server:", r.json()["count"])

Total features on server: 110292


In [ ]:
features = fetch_all_features(urls[0])
gdf = gpd.GeoDataFrame.from_features(features)

Fetched 2000 features...
Fetched 4000 features...
Fetched 6000 features...
Fetched 8000 features...
Fetched 10000 features...
Fetched 12000 features...
Fetched 14000 features...
Fetched 16000 features...
Fetched 18000 features...
Fetched 20000 features...
Fetched 22000 features...
Fetched 24000 features...
Fetched 26000 features...
Fetched 28000 features...
Fetched 30000 features...
Fetched 32000 features...
Fetched 34000 features...
Fetched 36000 features...
Fetched 38000 features...
Fetched 40000 features...
Fetched 42000 features...
Fetched 44000 features...
Fetched 46000 features...
Fetched 48000 features...
Fetched 50000 features...
Fetched 52000 features...
Fetched 54000 features...
Fetched 56000 features...
Fetched 58000 features...
Fetched 60000 features...
Fetched 62000 features...
Fetched 64000 features...
Fetched 66000 features...
Fetched 68000 features...
Fetched 70000 features...
Fetched 72000 features...
Fetched 74000 features...
Fetched 76000 features...
Fetched 78000 fe

In [ ]:
gdf.columns

Index(['geometry', 'FID', 'SPP', 'DBH', 'ADDRESS', 'STREET', 'SIDE', 'SITE',
       'Vacant'],
      dtype='object')

In [28]:
gdf['SPP'].unique()

array(['Zelkova serrata', 'Ulmus species', 'Zelkova  species',
       'Fraxinus americana', 'Fraxinus pennsylvanica', 'VACANT (GENERIC)',
       'vacant site medium', 'Acer negundo', 'Tilia cordata',
       'Acer campestre', 'vacant site small', 'Prunus serrulata',
       'Gleditsia triacanthos', 'Catalpa spp.', 'Crataegus x Lavallei',
       'Malus sylvestris', 'Pyrus calleryana', 'Quercus robur',
       'Acer platanoides', 'Acer saccharum', 'Stump',
       'Acer pseudoplatanus', 'Populus deltoides', 'Ulmus americana',
       'Pinus species', "Prunus virginiana 'Canada Red'",
       'Syringa reticulata', 'Acer x freemanii',
       'Koelreuteria paniculata', "Prunus x yedoensis 'Akebono'",
       "Morus alba 'Fruitless'", 'Aesculus hippocastanum',
       "Ulmus parvifolia 'Frontier'", 'Celtis occidentalis',
       'Celtis reticulata', 'Ailanthus altissima', 'Acer tataricum',
       'Prunus cerasifera', 'Parrotia persica',
       "Picea glauca 'Albertiana'", 'Acer saccharinum', 'Quercus

In [ ]:
gdf = gdf.set_crs("EPSG:4326")

In [ ]:
gdf = gdf.to_crs("EPSG:26912")
pixel_size = 0.6
# Get bounds
minx, miny, maxx, maxy = gdf.total_bounds

# Compute raster dimensions
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Define transform (origin = top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Prepare shapes for rasterization
shapes = ((geom, 1) for geom in gdf.geometry)

# Rasterize
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

/usr/local/lib/python3.12/dist-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 105815 will not be rasterized.
  warnings.warn(


In [ ]:
gdf.iloc[0]

,0
geometry,POINT (423833.0477826344 4512197.0141869)
FID,1
SPP,Zelkova serrata
DBH,10
ADDRESS,533
STREET,S 400 W [400 W]
SIDE,Front
SITE,1
Vacant,


In [ ]:
# Output file
output_path = "points_0_6m_full.tif"

# Write to GeoTIFF
with rasterio.open(
    output_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs="EPSG:26912",
    transform=transform,
) as dst:
    dst.write(raster, 1)

print("Raster written to:", output_path)

Raster written to: points_0_6m_full.tif
